In [ ]:
from pathlib import Path
import os
import re
import sys

import anndata as ad
import numpy as np
import pandas as pd
import torch


def find_project_root() -> Path:
    """Resolve the repository root when running from the root or analyses directory."""
    current_dir = Path.cwd().resolve()
    for candidate in (current_dir, current_dir.parent):
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the project root or the analyses directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "02_time_series_analysis"
RESULTS_DIR = PROJECT_ROOT / "results" / "02_time_series_analysis"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from scvolt.model import SCVOLTModel
from scvolt import tools

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
sc_expr = pd.read_csv(DATA_DIR / "GSE114461_Bortezomib.csv.gz").T


In [ ]:
latent_dim = 30
drug_name = "Bortezomib"
num_DRGs = 1200


In [ ]:
tools.set_model_seed(43)
data = {"sc": sc_expr}
model = SCVOLTModel(latent_dim=latent_dim, drug_name=drug_name, num_DRGs=num_DRGs)
encoded_data = model.encode(data=data, device=TORCH_DEVICE)
pred = model.predict(encoded_data=encoded_data)


In [ ]:
np.savetxt(RESULTS_DIR / "Bortezomib_score.csv", pred["sc_scores"], delimiter=",")


In [ ]:
# Predict all available CTRP drugs.
drug_dir = PROJECT_ROOT / "src" / "scvolt" / "resources" / "drgs" / "ctrp"
out_dir = RESULTS_DIR / "all_drug_scores"
out_dir.mkdir(parents=True, exist_ok=True)

drug_files = sorted(drug_dir.glob("*_signature.txt"))
drugs = [x.name[: -len("_signature.txt")] for x in drug_files]

all_scores = {}


def safe_name(name):
    """Return a filesystem-safe drug-name slug."""
    # Sanitize drug names before using them in file names.
    return re.sub(r'[\\/*?:"<>|]', "_", name)


for drug_name in drugs:
    print(f"Running {drug_name} ...")

    try:
        tools.set_model_seed(43)
        data = {"sc": sc_expr}

        model = SCVOLTModel(
            latent_dim=latent_dim,
            drug_name=drug_name,
            num_DRGs=num_DRGs,
        )

        encoded_data = model.encode(
            data=data,
            device=TORCH_DEVICE,
        )

        pred = model.predict(encoded_data=encoded_data)

        scores = pred["sc_scores"]

        if torch.is_tensor(scores):
            scores = scores.detach().cpu().numpy()

        scores = np.asarray(scores).reshape(-1)

        # Save the current drug's result separately.
        single_file = out_dir / f"{safe_name(drug_name)}.csv"
        pd.DataFrame({drug_name: scores}).to_csv(single_file, index=True)

        # Append the current result to the combined table.
        all_scores[drug_name] = scores

        pred_df = pd.DataFrame(all_scores)

        # Write to a temporary file first, then replace the final file atomically.
        tmp_file = out_dir / "all_drug_scores.tmp.csv"
        final_file = out_dir / "all_drug_scores.csv"

        pred_df.to_csv(tmp_file, index=True)
        os.replace(tmp_file, final_file)

        print(f"Finished {drug_name}")

    except Exception as e:
        print(f"Error in {drug_name}: {e}")
        continue
